# Fractional GPU allocations

Scope of the containers excluded from IR2 (0.x and 1.x-3.x GPU): counts, allocated GPU-time and CPU-time, and minute-trace coverage. Writes `05_Results/revision/04_fractional_*` (used for the 16.0% and 25.5% values in Section 4.5).

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()
from cache_paths import require_frozen_cache
ONLINE_CACHE = require_frozen_cache('online_ir2', ['container_events.csv'])

from cohort_contract import current_contract
LATEST_CONTRACT=current_contract()


## 1. Inputs and checks

In [ ]:
# Inputs and outputs stay inside the repository.
from pathlib import Path
from IPython.display import display as _cleanup_display, HTML as _CleanupHTML

RESULTS = RESULTS_REVISION
RESULTS.mkdir(parents=True, exist_ok=True)

def display(*objects, **kwargs):
    """Display-only terminology; dataframe values and calculation keys are unchanged."""
    for obj in objects:
        html = obj.data if isinstance(obj, _CleanupHTML) else (
            obj._repr_html_() if hasattr(obj, "_repr_html_") else None
        )
        if isinstance(html, str):
            html = html.replace("GPU-Enabled", "Integer-GPU")
            _cleanup_display(_CleanupHTML(html), **kwargs)
        else:
            _cleanup_display(obj, **kwargs)

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display as _cleanup_display

DATA_ROOT = Path(str(PROCESSED_DATA))
USAGE_PATH = DATA_ROOT / 'UsageLog20230401_20251130_20260301_20260831.csv'
TRACE_PATH = DATA_ROOT / 'TimeSeries_20240926_20251130_20260301_20260831.parquet'
assert USAGE_PATH.is_file() and TRACE_PATH.is_file()

font_path = Path('/System/Library/Fonts/AppleSDGothicNeo.ttc')
if font_path.is_file():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 120, 'font.size': 10})

def normalize_filename(series):
    return (series.astype('string').str.replace('\\', '/', regex=False)
            .str.split('/').str[-1].str.strip()
            .str.replace(r'\.(csv|parquet)$', '', regex=True, case=False))

def result_header(name, description, criterion):
    print(f'\n[RESULT] {name}\n설명: {description}\n판정 기준: {criterion}')

print('UsageLog:', USAGE_PATH)
print('TimeSeries:', TRACE_PATH)

# Cohort and applied IR2 timeouts (85/119/146/170) from the frozen cohort contract.
sys.path.insert(0, str(PROJECT_ROOT / "04_Analysis/revision"))
from cohort_contract import current_contract
LATEST_CONTRACT = current_contract()


### 2.1 UsageLog 분류와 자원시간 검증 / UsageLog classification

In [ ]:
usecols = [
    'filename', 'created_time', 'run_time', 'terminated_time', 'gpu_allocated(EA)',
    'gpu_allocated(group)', 'cpu_allocated(Core)', 'used_time(minute)',
    'gpu_util_avg(%)',
]
usage = pd.read_csv(USAGE_PATH, usecols=usecols, low_memory=False)
usage['filename_clean'] = normalize_filename(usage['filename'])
for col in ['created_time', 'run_time', 'terminated_time']:
    usage[col] = pd.to_datetime(usage[col], errors='coerce')
for col in ['gpu_allocated(EA)', 'cpu_allocated(Core)', 'used_time(minute)', 'gpu_util_avg(%)']:
    usage[col] = pd.to_numeric(usage[col], errors='coerce')

created = usage['created_time']
usage['period'] = np.select([
    created.ge(pd.Timestamp('2023-03-01')) & created.lt(pd.Timestamp('2025-12-01')),
    created.ge(pd.Timestamp('2026-03-01')) & created.lt(pd.Timestamp('2026-09-01')),
], ['Train / 학습', 'Test / 검증'], default='Outside / 대상 외')

allocated = usage['gpu_allocated(EA)']
integer = np.isclose(allocated, allocated.round(), rtol=0, atol=1e-9)
usage['gpu_class'] = np.select([
    allocated.eq(0),
    allocated.gt(0) & integer,
    allocated.gt(0) & allocated.lt(1),
    allocated.gt(1) & ~integer,
], [
    'Non-GPU / GPU 미할당',
    'Integer GPU / 정수 GPU',
    '0.x GPU / 1 미만 분할 GPU',
    '>1 noninteger / 1 초과 비정수 GPU',
], default='Invalid / 분류 불가')

usage['gpu_allocated_min'] = allocated * usage['used_time(minute)']
usage['cpu_allocated_min'] = usage['cpu_allocated(Core)'] * usage['used_time(minute)']
usage['elapsed_from_timestamps_min'] = (
    usage['terminated_time'] - usage['run_time']
).dt.total_seconds() / 60  # execution start to termination

group_ok = (
    usage.loc[usage['gpu_class'].eq('Non-GPU / GPU 미할당'), 'gpu_allocated(group)'].eq('0.0_GPU').all()
    and usage.loc[usage['gpu_class'].eq('0.x GPU / 1 미만 분할 GPU'), 'gpu_allocated(group)'].eq('0.x_GPU').all()
    and usage.loc[usage['gpu_class'].eq('>1 noninteger / 1 초과 비정수 GPU'), 'gpu_allocated(group)'].isna().all()
)
input_ok = (
    len(usage) == 39_853 and usage['filename_clean'].is_unique
    and usage['period'].ne('Outside / 대상 외').all()
    and usage['gpu_class'].ne('Invalid / 분류 불가').all()
    and usage[['created_time', 'terminated_time', 'gpu_allocated(EA)', 'cpu_allocated(Core)', 'used_time(minute)']].notna().all().all()
    and usage['used_time(minute)'].ge(0).all()
    and usage['cpu_allocated(Core)'].ge(0).all()
    and np.allclose(usage['used_time(minute)'], usage['elapsed_from_timestamps_min'], rtol=0, atol=1e-7)
    and group_ok
)

result_header('fractional_usage_input', 'UsageLog 컨테이너 단위·기간·할당 그룹·수명 확인',
              '39,853개 고유 컨테이너, GPU/CPU allocation 필수값·음수·기간 외 0, 종료−실행시작 시간과 used_time 일치')
print('Result:', 'PASS' if input_ok else 'FAIL')
if not input_ok:
    raise ValueError('UsageLog 입력 검증 실패')


### 결과
### 2.2 분 단위 관측 범위 / Minute-level trace coverage

In [ ]:
parquet = pq.ParquetFile(TRACE_PATH)
metadata = parquet.metadata
name_to_column = {name: index for index, name in enumerate(parquet.schema_arrow.names)}
required = ['filename', 'gpu_allocated(group)', 'used_time(minute)']
assert all(name in name_to_column for name in required)

trace_rows = []
for index in range(metadata.num_row_groups):
    row_group = metadata.row_group(index)
    file_stat = row_group.column(name_to_column['filename']).statistics
    group_stat = row_group.column(name_to_column['gpu_allocated(group)']).statistics
    elapsed_stat = row_group.column(name_to_column['used_time(minute)']).statistics
    if (file_stat is None or group_stat is None or elapsed_stat is None
            or file_stat.min != file_stat.max or group_stat.min != group_stat.max
            or row_group.num_rows <= 0):
        raise ValueError(f'Row group {index}가 단일 컨테이너·할당 그룹 전제를 위반합니다.')
    trace_rows.append((file_stat.min, group_stat.min, float(elapsed_stat.max), row_group.num_rows))

trace = pd.DataFrame(trace_rows, columns=[
    'filename_clean', 'trace_gpu_group', 'trace_end_minute', 'trace_sample_count'
])
trace['filename_clean'] = normalize_filename(trace['filename_clean'])
trace_ok = (
    len(trace) == LATEST_CONTRACT['row_groups'] and trace['filename_clean'].is_unique
    and trace['trace_end_minute'].ge(0).all()
    and set(trace['filename_clean']).issubset(set(usage['filename_clean']))
)
joined = usage.merge(trace, on='filename_clean', how='left', validate='one_to_one', indicator=True)
joined['has_minute_trace'] = joined['_merge'].eq('both')
joined['trace_gpu_allocated_min'] = joined['gpu_allocated(EA)'] * joined['trace_end_minute']
same_group = joined.loc[joined['has_minute_trace'], 'gpu_allocated(group)'].eq(
    joined.loc[joined['has_minute_trace'], 'trace_gpu_group']
).all()
trace_ok &= bool(same_group)

result_header('fractional_trace_coverage', 'Parquet row-group 통계와 UsageLog의 컨테이너별 결합',
              '15,125개 고유 trace·미연결 0·할당 그룹 일치')
print('TimeSeries rows / 관측 행:', metadata.num_rows)
print('TimeSeries containers / 관측 컨테이너:', len(trace))
print('Result:', 'PASS' if trace_ok else 'FAIL')
if not trace_ok:
    raise ValueError('TimeSeries 관측 범위 검증 실패')


### 결과
## 3. 제외 집단의 규모 / Results: excluded allocation-time

In [ ]:
category_order = [
    'Non-GPU / GPU 미할당', 'Integer GPU / 정수 GPU',
    '0.x GPU / 1 미만 분할 GPU', '>1 noninteger / 1 초과 비정수 GPU',
]
period_order = ['Train / 학습', 'Test / 검증']

summary = (
    joined.groupby(['period', 'gpu_class'], observed=True)
    .agg(containers=('filename_clean', 'size'),
         allocated_gpu_min=('gpu_allocated_min', 'sum'),
         allocated_cpu_min=('cpu_allocated_min', 'sum'),
         trace_containers=('has_minute_trace', 'sum'),
         trace_gpu_min=('trace_gpu_allocated_min', 'sum'))
    .reindex(pd.MultiIndex.from_product([period_order, category_order], names=['period', 'gpu_class']), fill_value=0)
)
summary['trace_coverage_pct'] = np.where(
    summary['containers'] > 0, summary['trace_containers'] / summary['containers'] * 100, np.nan
)

print('[RESULT] fractional_population_and_capacity')
print('표 A / Table A: 생성 cohort의 할당 자원시간과 분 단위 관측 범위')
display(summary.rename(columns={
    'containers': 'Containers / 컨테이너 수',
    'allocated_gpu_min': 'Allocated GPU-min / 할당 GPU·분',
    'allocated_cpu_min': 'Allocated CPU-min / 할당 CPU·분',
    'trace_containers': 'Traced / 분 단위 관측 수',
    'trace_gpu_min': 'Trace GPU-min / 관측 subset GPU·분',
    'trace_coverage_pct': 'Trace coverage % / 컨테이너 관측률 %',
}).round(2))

comparison_rows = []
for period in period_order:
    data = joined.loc[joined['period'].eq(period)]
    gpu = data.loc[data['gpu_allocated(EA)'].gt(0)]

    # 원고 Table 3의 GPU 분모: Integer-GPU + 0.x, >1 noninteger 제외
    paper_like = gpu.loc[gpu['gpu_class'].ne('>1 noninteger / 1 초과 비정수 GPU')]
    below_one = gpu.loc[gpu['gpu_class'].eq('0.x GPU / 1 미만 분할 GPU')]
    all_noninteger = gpu.loc[gpu['gpu_class'].isin([
        '0.x GPU / 1 미만 분할 GPU', '>1 noninteger / 1 초과 비정수 GPU'])]

    # CPU headline coverage 진단용 분모: Non-GPU + Integer-GPU + 0.x, >1 noninteger 제외
    # 기존 CPU All은 Non-GPU + Integer-GPU만 포함한다.
    cpu_scope = data.loc[data['gpu_class'].ne('>1 noninteger / 1 초과 비정수 GPU')]

    comparison_rows.append({
        'Period / 기간': period,
        'GPU containers, Table 3 scope / GPU 컨테이너 수': len(paper_like),
        '0.x containers / 0.x 컨테이너 수': len(below_one),
        '0.x count share % / 개수 비중 %': len(below_one) / len(paper_like) * 100,
        '0.x allocated GPU-min / 0.x 할당 GPU·분': below_one['gpu_allocated_min'].sum(),
        '0.x GPU-min share % / GPU 자원시간 비중 %': below_one['gpu_allocated_min'].sum() / paper_like['gpu_allocated_min'].sum() * 100,
        '0.x allocated CPU-min / 0.x 할당 CPU·분': below_one['cpu_allocated_min'].sum(),
        '0.x CPU-min share % / CPU scope 자원시간 비중 %': below_one['cpu_allocated_min'].sum() / cpu_scope['cpu_allocated_min'].sum() * 100,
        'All noninteger GPU-min share % / 전체 비정수 GPU 자원시간 비중 %': all_noninteger['gpu_allocated_min'].sum() / gpu['gpu_allocated_min'].sum() * 100,
        'All noninteger CPU-min share % / 전체 비정수 CPU 자원시간 비중 %': all_noninteger['cpu_allocated_min'].sum() / data['cpu_allocated_min'].sum() * 100,
    })
comparison = pd.DataFrame(comparison_rows).set_index('Period / 기간')
print('표 B / Table B: 0.x exclusion의 count·GPU-time·CPU-time scope')
display(comparison.round(2))

paper_check = (
    len(joined.loc[joined['period'].eq('Train / 학습') & joined['gpu_class'].ne('>1 noninteger / 1 초과 비정수 GPU')]) == 33_204
    and summary.loc[('Train / 학습', '0.x GPU / 1 미만 분할 GPU'), 'containers'] == 5_902
    and comparison.loc['Train / 학습', 'GPU containers, Table 3 scope / GPU 컨테이너 수'] == 16_749
)
baseline_check = (
    np.isclose(summary.loc[('Train / 학습', 'Integer GPU / 정수 GPU'), 'trace_gpu_min'], LATEST_CONTRACT['resources']['Train']['GPU']['baseline'], atol=.1)
    and np.isclose(summary.loc[('Test / 검증', 'Integer GPU / 정수 GPU'), 'trace_gpu_min'], LATEST_CONTRACT['resources']['Test']['GPU']['baseline'], atol=.1)
)
excluded_trace_check = (
    summary.loc[(slice(None), ['0.x GPU / 1 미만 분할 GPU', '>1 noninteger / 1 초과 비정수 GPU']), 'trace_containers'].eq(0).all()
)
cpu_scope_check = (
    summary['allocated_cpu_min'].ge(0).all()
    and comparison['0.x allocated CPU-min / 0.x 할당 CPU·분'].ge(0).all()
    and comparison['0.x CPU-min share % / CPU scope 자원시간 비중 %'].between(0, 100).all()
    and comparison['All noninteger CPU-min share % / 전체 비정수 CPU 자원시간 비중 %'].between(0, 100).all()
)
print('Table 3 재현 / reproduction:', bool(paper_check))
print('기존 분석 GPU 분모 재현 / denominator:', bool(baseline_check))
print('비정수 GPU 분 단위 관측 0 / excluded traces:', bool(excluded_trace_check))
print('CPU allocation scope 검산 / CPU scope:', bool(cpu_scope_check))
print('Result:', 'PASS' if paper_check and baseline_check and excluded_trace_check and cpu_scope_check else 'FAIL')
if not (paper_check and baseline_check and excluded_trace_check and cpu_scope_check):
    raise ValueError('모집단·분모·CPU scope 검산 실패')


### 결과
### 3.1 Fractional 제외 후 IR2 절감률 확인 / Eligible-subset savings rates

In [ ]:
EVENTS_PATH = ONLINE_CACHE / 'container_events.csv'
assert EVENTS_PATH.is_file(), f'사후 평가 사건 파일이 없습니다: {EVENTS_PATH}'
events = pd.read_csv(EVENTS_PATH, usecols=[
    'filename', 'population', 'gpu_allocated', 'cpu_allocated',
    'baseline_min', 'hindsight_saved_min',
])
events['filename_clean'] = normalize_filename(events['filename'])
events_with_class = events.merge(
    joined[['filename_clean', 'period', 'gpu_class', 'has_minute_trace']],
    on='filename_clean', how='left', validate='one_to_one', indicator=True,
)
eligible_classes = {'Non-GPU / GPU 미할당', 'Integer GPU / 정수 GPU'}
eligible_check = (
    len(events_with_class) == LATEST_CONTRACT['row_groups']
    and events_with_class['filename_clean'].is_unique
    and events_with_class['_merge'].eq('both').all()
    and events_with_class['has_minute_trace'].all()
    and events_with_class['gpu_class'].isin(eligible_classes).all()
    and events_with_class['period'].eq(events_with_class['population'].map({
        'Train': 'Train / 학습', 'Test': 'Test / 검증',
    })).all()
)

rate_rows = []
for population, label in [('Train', 'Train / 학습'), ('Test', 'Test / 검증')]:
    cohort = events_with_class.loc[events_with_class['population'].eq(population)]
    for resource, allocation_col in [('GPU', 'gpu_allocated'), ('CPU', 'cpu_allocated')]:
        eligible = cohort.loc[cohort[allocation_col].gt(0)]
        baseline = float((eligible[allocation_col] * eligible['baseline_min']).sum())
        saved = float((eligible[allocation_col] * eligible['hindsight_saved_min']).sum())
        rate_rows.append({
            'Period / 기간': label,
            'Resource / 자원': f'{resource} / {"GPU" if resource == "GPU" else "CPU 코어"}',
            'Eligible containers / 절감률 포함 컨테이너': len(eligible),
            'Allocated resource-min / 관측 집단 할당 자원·분': baseline,
            'Pre+Post saved resource-min / 사후 절감 자원·분': saved,
            'Saved rate % / 사후 절감률 %': saved / baseline * 100,
        })
rates = pd.DataFrame(rate_rows).set_index(['Period / 기간', 'Resource / 자원'])
expected = {(label,f'{res} / {"GPU" if res=="GPU" else "CPU 코어"}'):
    (LATEST_CONTRACT['resources'][pop][res]['baseline'],LATEST_CONTRACT['resources'][pop][res]['total_saved'])
    for pop,label in [('Train','Train / 학습'),('Test','Test / 검증')] for res in ['GPU','CPU']}
rate_check = all(
    np.allclose(rates.loc[key, [
        'Allocated resource-min / 관측 집단 할당 자원·분',
        'Pre+Post saved resource-min / 사후 절감 자원·분',
    ]].to_numpy(dtype=float), values, rtol=0, atol=.1)
    for key, values in expected.items()
)
result_header('fractional_excluded_savings_rate',
              'Fractional GPU 제외 후 동일 적격 집단의 IR2 사후 절감률 재계산',
              '15,125개 사건이 분 단위 관측 Non-GPU·정수 GPU와 일치하고 최초 투고·Test 합계 재현')
print('표 C / Table C: fractional GPU 제외 집단의 GPU·CPU 사후 절감률')
display(rates.style.format({
    'Eligible containers / 절감률 포함 컨테이너': '{:,.0f}',
    'Allocated resource-min / 관측 집단 할당 자원·분': '{:,.2f}',
    'Pre+Post saved resource-min / 사후 절감 자원·분': '{:,.2f}',
    'Saved rate % / 사후 절감률 %': '{:.2f}',
}))
print('Result:', 'PASS' if eligible_check and rate_check else 'FAIL')
if not (eligible_check and rate_check):
    raise ValueError('IR2 절감률 적격 집단 또는 합계 검산 실패')


### 결과
### 3.2 개수 비율과 자원시간 비율 / Count share versus capacity share

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
x = np.arange(2)
width = 0.34
count_pct = comparison['0.x count share % / 개수 비중 %'].to_numpy()
minute_pct = comparison['0.x GPU-min share % / GPU 자원시간 비중 %'].to_numpy()
bars_count = ax.bar(x - width/2, count_pct, width, color='#255B8A', label='Count share / 컨테이너 수 비중')
bars_minutes = ax.bar(x + width/2, minute_pct, width, color='#D89C32', label='GPU-minute share / 할당 GPU·분 비중')
for bars in [bars_count, bars_minutes]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + .7,
                f'{bar.get_height():.2f}%', ha='center', va='bottom', fontsize=10)
ax.set_xticks(x, period_order)
ax.set_ylim(0, max(count_pct.max(), minute_pct.max()) + 8)
ax.set_ylabel('Share of integer + 0.x GPU cohort (%) / 유형별 비중 (%)')
ax.set_title('0.x GPU: container count vs allocated GPU-minutes / 개수와 할당 자원시간')
ax.grid(axis='y', color='#D9DEE4', linewidth=.7)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(.5, -.17), ncol=2, fontsize=9)
fig.subplots_adjust(left=.12, right=.98, top=.89, bottom=.25)
fig.savefig(RESULTS / "04_fractional_scope.png", dpi=180, bbox_inches="tight")
from IPython.display import Image
display(Image(filename=str(RESULTS / "04_fractional_scope.png")))
plt.close(fig)
print('해석: 35.24%는 Train의 컨테이너 수 비중이며, 같은 범위의 자원시간 비중은 17.37%다.')


## 4. 이용률 지표 적합성 / Metric fitness and uncertainty

In [ ]:
quality = (
    joined.loc[joined['gpu_allocated(EA)'].gt(0)]
    .groupby(['period', 'gpu_class'], observed=True)
    .agg(containers=('filename_clean', 'size'),
         avg_util_present=('gpu_util_avg(%)', 'count'),
         avg_util_above_100=('gpu_util_avg(%)', lambda s: int(s.gt(100).sum())),
         minute_traces=('has_minute_trace', 'sum'))
    .reindex(pd.MultiIndex.from_product([
        period_order, category_order[1:]], names=['period', 'gpu_class']), fill_value=0)
)
quality['avg_util_available_pct'] = quality['avg_util_present'] / quality['containers'] * 100
print('[RESULT] fractional_metric_fitness')
print('표 D / Table D: GPU 평균 이용률 필드의 존재와 100% 초과값')
display(quality.rename(columns={
    'containers': 'Containers / 컨테이너 수',
    'avg_util_present': 'Average util present / 평균 이용률 기입 수',
    'avg_util_above_100': 'Above 100% / 100% 초과 수',
    'minute_traces': 'Minute traces / 분 단위 관측 수',
    'avg_util_available_pct': 'Available % / 기입률 %',
}).round(2))

assert quality.loc[('Train / 학습', '0.x GPU / 1 미만 분할 GPU'), 'avg_util_above_100'] == 18
assert quality.loc[('Test / 검증', '0.x GPU / 1 미만 분할 GPU'), 'avg_util_above_100'] == 4
print('Result: PASS (필드 상태 재현; 지표의 컨테이너별 귀속 타당성은 미검증)')


In [ ]:
# Deterministic exports of retained summary tables.
# summary/comparison also retain allocated CPU-minutes and CPU scope diagnostics;
# fractional reclaimable savings are still not estimated.
summary.to_csv(RESULTS / "04_fractional_allocation_scope.csv", index=True)
comparison.to_csv(RESULTS / "04_fractional_scope_comparison.csv", index=True)
quality.to_csv(RESULTS / "04_fractional_metric_quality.csv", index=True)
